# Salarios del IMSS: 22.8 millones de puestos de trabajo en cinco archivos abiertos

Las vacantes en México casi nunca publican el sueldo. El IMSS sí lo registra: cada mes publica el salario base de cotización de cada puesto de trabajo afiliado, agregado por estado, municipio, sector, sexo, edad, rango salarial y tamaño del patrón. Este notebook descarga cinco cortes de agosto, de 2018 a 2026, documenta el formato y deja agregados compactos con los que trabaja el notebook 02.

**Fuente:** Datos Abiertos IMSS, conjuntos `asg-<año>` (licencia Libre Uso MX). Archivos mensuales de 300 a 400 MB y cerca de cinco millones de filas cada uno. Diccionario, glosario y preguntas frecuentes oficiales en `docs/fuente/`.

## 1. Preparación

Cinco cortes de agosto, cada dos años. Agosto es el mes del último boletín publicado y evita la estacionalidad de diciembre. Polars lee los archivos sin cargarlos completos en memoria.

In [1]:
import json
import time
from datetime import date
from pathlib import Path

import pandas as pd
import polars as pl
import requests

RAIZ = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA = RAIZ / "data"
RAW = DATA / "raw"
RAW.mkdir(parents=True, exist_ok=True)
DICCIONARIO = RAIZ / "docs" / "fuente" / "diccionario_de_datos_asegurados.xlsx"

ANIOS = [2018, 2020, 2022, 2024, 2026]
CORTE = "08-31"
BASE = "http://datos.imss.gob.mx/sites/default/files"
API = "http://datos.imss.gob.mx/api/3/action/package_show"
UA = (
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/128.0 Safari/537.36"
)
SESION = requests.Session()
SESION.headers["User-Agent"] = UA
pl.Config.set_tbl_rows(40)
pl.Config.set_fmt_str_lengths(60)
print("Crudos en:", RAW)

Crudos en: /Users/mauvilar/Desktop/DA/Proyectos/salarios-imss-mx/data/raw


## 2. El catálogo y la descarga

El portal corre DKAN y expone la misma API que CKAN: `package_show` lista los archivos de cada año con su tamaño. Solo responde por `http` y con un `User-Agent` de navegador. La descarga guarda en `data/raw/` y no repite lo que ya está.

In [2]:
try:
    catalogo = SESION.get(API, params={"id": "asg-2026"}, timeout=60).json()["result"]
    catalogo = catalogo[0] if isinstance(catalogo, list) else catalogo
    display(pd.DataFrame([(r["name"], r["format"], r["size"], r["url"]) for r in catalogo["resources"]], columns=["archivo", "formato", "tamaño", "url"]))
except (requests.RequestException, ValueError, KeyError) as e:
    # El portal del IMSS se cae con frecuencia (503 o HTML en vez de JSON). Los archivos ya descargados siguen en data/raw/.
    print(f"El catálogo no respondió ({type(e).__name__}); se sigue con los archivos en caché.")

El catálogo no respondió (JSONDecodeError); se sigue con los archivos en caché.


In [3]:
def descargar(anio: int) -> Path:
    """Descarga el corte de agosto de un año a data/raw/, si no está ya."""
    destino = RAW / f"asg-{anio}-{CORTE}.csv"
    if destino.exists() and destino.stat().st_size > 100_000_000:
        return destino
    parcial = destino.with_suffix(".parcial")
    with SESION.get(f"{BASE}/asg-{anio}-{CORTE}.csv", stream=True, timeout=900) as r:
        r.raise_for_status()
        with open(parcial, "wb") as f:
            for trozo in r.iter_content(chunk_size=1 << 20):
                f.write(trozo)
    parcial.rename(destino)
    return destino


archivos: dict[int, Path] = {}
for anio in ANIOS:
    inicio = time.perf_counter()
    archivos[anio] = descargar(anio)
    print(f"{archivos[anio].name}  {archivos[anio].stat().st_size / 1e6:6.0f} MB  {time.perf_counter() - inicio:5.1f} s")

asg-2018-08-31.csv     388 MB    0.0 s
asg-2020-08-31.csv     368 MB    0.0 s
asg-2022-08-31.csv     405 MB    0.0 s
asg-2024-08-31.csv     386 MB    0.0 s
asg-2026-08-31.csv     403 MB    0.0 s


## 3. Esquema por año

Separador `|`, codificación latin-1 y la palabra `NA` donde no aplica. Las columnas cambian con los años: `ptpd` (trabajador de plataforma digital) aparece en los archivos recientes.

In [4]:
def abrir(anio: int) -> pl.LazyFrame:
    """Lectura perezosa de un corte. Renombra la columna con la ñ rota."""
    lf = pl.scan_csv(
        archivos[anio], separator="|", encoding="utf8-lossy", null_values=["NA"], infer_schema_length=10_000
    )
    columnas = lf.collect_schema().names()
    rota = next((c for c in columnas if c.startswith("tama")), None)
    return lf.rename({rota: "tamano_patron"}) if rota and rota != "tamano_patron" else lf


esquemas = {anio: abrir(anio).collect_schema().names() for anio in ANIOS}
todas = sorted({c for cols in esquemas.values() for c in cols}, key=lambda c: esquemas[2026].index(c) if c in esquemas[2026] else 99)
pd.DataFrame({anio: [c in cols for c in todas] for anio, cols in esquemas.items()}, index=todas).replace({True: "sí", False: ""})

,2018,2020,2022,2024,2026
cve_delegacion,sí,sí,sí,sí,sí
cve_subdelegacion,sí,sí,sí,sí,sí
cve_entidad,sí,sí,sí,sí,sí
cve_municipio,sí,sí,sí,sí,sí
ptpd,,,,,sí
sector_economico_1,sí,sí,sí,sí,sí
sector_economico_2,sí,sí,sí,sí,sí
sector_economico_4,sí,sí,sí,sí,sí
tamano_patron,sí,sí,sí,sí,sí
sexo,sí,sí,sí,sí,sí


## 4. Cinco mañas del formato

**Maña 1 · Dos poblaciones en una tabla.** `asegurados` incluye a los `no_trabajadores`, asegurados sin un empleo asociado (estudiantes, seguro facultativo, continuación voluntaria). Los puestos de trabajo son `ta`; el salario solo existe para `ta_sal`. Sumar `asegurados` como si fueran empleos infla el empleo formal en millones.

In [5]:
totales_2026 = (
    abrir(2026)
    .select(
        pl.col("asegurados").sum(),
        pl.col("no_trabajadores").sum(),
        pl.col("ta").sum(),
        pl.col("ta_sal").sum(),
        pl.col("masa_sal_ta").sum(),
    )
    .collect()
)
totales_2026

asegurados,no_trabajadores,ta,ta_sal,masa_sal_ta
i64,i64,i64,i64,f64
30905819,8097989,22798473,22739009,1.5305e10


**Maña 2 · El salario promedio es masa salarial entre puestos con salario.** `masa_sal_ta / ta_sal`, en pesos diarios. Dividir entre `ta` lo subestima porque hay puestos sin salario asociado.

In [6]:
t = totales_2026.row(0, named=True)
print(f"masa / ta_sal = {t['masa_sal_ta'] / t['ta_sal']:.2f} pesos diarios   |   masa / ta = {t['masa_sal_ta'] / t['ta']:.2f}")

masa / ta_sal = 673.05 pesos diarios   |   masa / ta = 671.30


**Maña 3 · El salario está topado a 25 UMA.** La Ley del Seguro Social fija el salario base de cotización máximo en 25 veces la Unidad de Medida y Actualización. Quien gana más cotiza exactamente el tope, y en los datos aparece en el rango `W25` de `rango_uma`. Como el salario mínimo creció más rápido que la UMA, el tope pasó de 22.8 salarios mínimos en 2018 a 9.3 en 2026: los topados caen en `W23` del rango salarial en 2018 y en `W10` en 2026. Consecuencia: el promedio es una **cota inferior** del salario real y la distribución por múltiplos del mínimo se corta en el tope. La mediana y los cuartiles no se ven afectados, porque el tope queda muy por arriba de ellos.

In [7]:
UMA = {2018: 80.60, 2020: 86.88, 2022: 96.22, 2024: 108.57, 2026: 117.31}  # pesos diarios, INEGI, vigente en agosto de cada año
SALARIO_MINIMO = {2018: 88.36, 2020: 123.22, 2022: 172.87, 2024: 248.93, 2026: 315.04}  # CONASAMI, zona general
filas = []
for anio in ANIOS:
    df = abrir(anio).select(["rango_uma", "rango_salarial", "ta_sal", "masa_sal_ta"]).collect()
    total, masa = df["ta_sal"].sum(), df["masa_sal_ta"].sum()
    topados = df.filter(pl.col("rango_uma") == "W25")
    rango_sm = topados.group_by("rango_salarial").agg(pl.col("ta_sal").sum()).sort("ta_sal", descending=True)["rango_salarial"][0]
    filas.append({
        "anio": anio, "uma": UMA[anio], "tope_pesos": 25 * UMA[anio],
        "tope_en_salarios_minimos": round(25 * UMA[anio] / SALARIO_MINIMO[anio], 1),
        "puestos_topados": topados["ta_sal"].sum(), "pct_puestos": round(100 * topados["ta_sal"].sum() / total, 2),
        "pct_masa_salarial": round(100 * topados["masa_sal_ta"].sum() / masa, 2), "rango_sm_donde_caen": rango_sm,
    })
tope = pd.DataFrame(filas).set_index("anio")
tope

,uma,tope_pesos,tope_en_salarios_minimos,puestos_topados,pct_puestos,pct_masa_salarial,rango_sm_donde_caen
anio,,,,,,,
2018,80.60,2015.00,22.8,371711,1.87,10.54,W23
2020,86.88,2172.00,17.6,381764,1.96,10.52,W18
2022,96.22,2405.50,13.9,462845,2.19,10.87,W14
2024,108.57,2714.25,10.9,487251,2.19,10.08,W11
2026,117.31,2932.75,9.3,526570,2.32,10.07,W10


**Maña 4 · Catálogos propios.** Las claves de entidad son del IMSS y los sectores usan una división de 1990 con "2 - 3" para una sola división (industrias de la transformación). Los nombres se leen del diccionario oficial, no se escriben a mano.

In [8]:
entidades = (
    pd.read_excel(DICCIONARIO, sheet_name="entidad-municipio", header=1)
    .rename(columns=lambda c: str(c).strip())
    .loc[:, ["cve_entidad", "descripción entidad"]]
    .drop_duplicates()
    .assign(cve_entidad=lambda d: d["cve_entidad"].astype(int))
    .rename(columns={"descripción entidad": "entidad"})
    .sort_values("cve_entidad")
)
sectores = pd.read_excel(DICCIONARIO, sheet_name="sector 1", header=1)
sectores.columns = ["sector_economico_1", "sector"]
sectores = sectores.dropna()
sectores["sector"] = sectores["sector"].str.replace("Div-", "", regex=False).str.strip()
filas = []
for _, r in sectores.iterrows():
    for codigo in str(r["sector_economico_1"]).split("-"):
        filas.append((int(codigo), r["sector"]))
sectores = pd.DataFrame(filas, columns=["sector_economico_1", "sector"])
print(len(entidades), "entidades ·", len(sectores), "códigos de sector")
sectores

32 entidades · 10 códigos de sector


,sector_economico_1,sector
0,0,"Agricultura, Ganadería, Silvicultura, Pesca y ..."
1,1,Industrias Extractivas
2,2,Industrias de la Transformación
3,3,Industrias de la Transformación
4,4,Industria de la Construcción
5,5,Ind Eléctrica y Captación y Suministro de Agua...
6,6,Comercio
7,7,Transportes y Comunicaciones
8,8,"Servicios para Empresas, Personas y el Hogar"
9,9,Servicios Sociales y Comunales


**Maña 5 · Sexo con tres valores y edad en rangos.** El catálogo trae 1 hombre, 2 mujer y 3 no binario; el 3 tiene tan pocos puestos que no se grafica, pero sí se cuenta. La edad viene en rangos de cinco años (`E1` a `E14`).

In [9]:
abrir(2026).group_by("sexo").agg(pl.col("ta").sum(), pl.col("ta_sal").sum(), pl.col("masa_sal_ta").sum()).collect().sort("sexo")

sexo,ta,ta_sal,masa_sal_ta
i64,i64,i64,f64
1,13585251,13547308,9.5312e9
2,9213146,9191625,5.7733e9
3,76,76,50137.35


## 5. Agregar

Diez agregados por año. Cada uno conserva `ta`, `ta_sal`, `masa_sal_ta` y `permanentes` para que el notebook 02 calcule promedios y participaciones sin volver a los crudos. Si una dimensión no existe en un año, la columna va vacía y se reporta.

In [10]:
AGREGADOS = {
    "nacional": [],
    "entidad_sexo": ["cve_entidad", "sexo"],
    "sector_sexo": ["sector_economico_1", "sexo"],
    "rango_salarial": ["rango_salarial"],
    "rango_uma": ["rango_uma"],
    "entidad_rango": ["cve_entidad", "rango_salarial"],
    "entidad_uma": ["cve_entidad", "rango_uma"],
    "sexo_rango": ["sexo", "rango_salarial"],
    "tamano_patron": ["tamano_patron"],
    "edad_sexo": ["rango_edad", "sexo"],
    "plataforma": ["ptpd"],
}
METRICAS = [
    pl.col("ta").sum(),
    pl.col("ta_sal").sum(),
    pl.col("masa_sal_ta").sum(),
    (pl.col("tpu") + pl.col("tpc")).sum().alias("permanentes"),
    pl.col("no_trabajadores").sum(),
]


def agregar(anio: int, dims: list[str]) -> pl.DataFrame:
    lf = abrir(anio)
    presentes = [d for d in dims if d in lf.collect_schema().names()]
    if presentes:
        df = lf.group_by(presentes).agg(METRICAS).collect()
    else:
        df = lf.select(METRICAS).collect()
    for d in dims:
        if d not in presentes:
            df = df.with_columns(pl.lit(None, dtype=pl.Int64).alias(d))
    return df.with_columns(pl.lit(anio).alias("anio")).select(["anio", *dims, "ta", "ta_sal", "masa_sal_ta", "permanentes", "no_trabajadores"])


agregados: dict[str, pl.DataFrame] = {}
inicio = time.perf_counter()
for nombre, dims in AGREGADOS.items():
    agregados[nombre] = pl.concat([agregar(anio, dims) for anio in ANIOS])
print(f"{sum(len(v) for v in agregados.values()):,} filas agregadas en {time.perf_counter() - inicio:.1f} s")
faltantes = {nombre: [a for a in ANIOS if agregados[nombre].filter(pl.col("anio") == a)[dims[0]].null_count() == agregados[nombre].filter(pl.col("anio") == a).height] for nombre, dims in AGREGADOS.items() if dims}
print("dimensiones ausentes por año:", {k: v for k, v in faltantes.items() if v})
agregados["nacional"]

7,710 filas agregadas en 15.5 s
dimensiones ausentes por año: {'plataforma': [2018, 2020, 2022, 2024]}


anio,ta,ta_sal,masa_sal_ta,permanentes,no_trabajadores
i32,i64,i64,f64,i64,i64
2018,20063433,19927946,7.0973e9,17182201,7101201
2020,19588342,19464360,7.8661e9,16909339,7733219
2022,21236866,21124599,1.0232e10,18414515,7851341
2024,22389835,22299547,1.3100e10,19383168,8090500
2026,22798473,22739009,1.5305e10,19860128,8097989


## 6. Validaciones

Tres aserciones. Si alguna falla, el snapshot no se escribe.

**V1 · Identidad interna en los cinco años:** los puestos afiliados son la suma de permanentes y eventuales, urbanos y del campo.

In [11]:
identidad = pl.concat(
    [abrir(a).select(pl.lit(a).alias("anio"), pl.col("ta").sum(), (pl.col("tpu") + pl.col("tpc") + pl.col("teu") + pl.col("tec")).sum().alias("suma")).collect() for a in ANIOS]
)
print(identidad)
assert (identidad["ta"] == identidad["suma"]).all(), "ta no coincide con la suma de sus componentes"

shape: (5, 3)
┌──────┬──────────┬──────────┐
│ anio ┆ ta       ┆ suma     │
│ ---  ┆ ---      ┆ ---      │
│ i32  ┆ i64      ┆ i64      │
╞══════╪══════════╪══════════╡
│ 2018 ┆ 20063433 ┆ 20063433 │
│ 2020 ┆ 19588342 ┆ 19588342 │
│ 2022 ┆ 21236866 ┆ 21236866 │
│ 2024 ┆ 22389835 ┆ 22389835 │
│ 2026 ┆ 22798473 ┆ 22798473 │
└──────┴──────────┴──────────┘


**V2 · Agosto de 2026 contra el boletín del IMSS** (comunicado del 8 de septiembre de 2026): 22,798,473 puestos, salario base de cotización promedio de 673.1 pesos diarios, 87.1 % permanentes.

In [12]:
n = agregados["nacional"].filter(pl.col("anio") == 2026).row(0, named=True)
sbc = n["masa_sal_ta"] / n["ta_sal"]
permanentes = 100 * n["permanentes"] / n["ta"]
print(f"puestos {n['ta']:,}  ·  SBC {sbc:.2f}  ·  permanentes {permanentes:.1f} %")
assert n["ta"] == 22_798_473, "los puestos no coinciden con el boletín"
assert abs(sbc - 673.1) < 0.1, "el SBC no coincide con el boletín"
assert abs(permanentes - 87.1) < 0.1, "la proporción de permanentes no coincide con el boletín"

puestos 22,798,473  ·  SBC 673.05  ·  permanentes 87.1 %


**V3 · Los agregados suman lo mismo que el nacional**, año por año, para cada dimensión.

In [13]:
for nombre, df in agregados.items():
    if nombre == "nacional":
        continue
    por_anio = df.group_by("anio").agg(pl.col("ta").sum(), pl.col("masa_sal_ta").sum()).sort("anio")
    nacional = agregados["nacional"].select("anio", "ta", "masa_sal_ta").sort("anio")
    assert (por_anio["ta"] == nacional["ta"]).all(), f"{nombre}: ta no cierra"
    assert ((por_anio["masa_sal_ta"] - nacional["masa_sal_ta"]).abs() < 1).all(), f"{nombre}: masa no cierra"
    print(f"{nombre:16s} cierra en los {len(por_anio)} años")

entidad_sexo     cierra en los 5 años
sector_sexo      cierra en los 5 años
rango_salarial   cierra en los 5 años
rango_uma        cierra en los 5 años
entidad_rango    cierra en los 5 años
entidad_uma      cierra en los 5 años
sexo_rango       cierra en los 5 años
tamano_patron    cierra en los 5 años
edad_sexo        cierra en los 5 años
plataforma       cierra en los 5 años


## 7. Snapshot

Los agregados se guardan como CSV en `data/` con los nombres de entidad y sector ya pegados, y un manifiesto con la fecha, los archivos crudos y sus tamaños. Los crudos no se versionan.

In [14]:
ent = pl.from_pandas(entidades)
sec = pl.from_pandas(sectores)
salidas = dict(agregados)
for nombre in ["entidad_sexo", "entidad_rango", "entidad_uma"]:
    salidas[nombre] = salidas[nombre].join(ent, on="cve_entidad", how="left")
salidas["sector_sexo"] = salidas["sector_sexo"].join(sec, on="sector_economico_1", how="left")
manifiesto = {
    "fecha_consulta": date.today().isoformat(),
    "fuente": "Datos Abiertos IMSS · conjuntos asg-<año> · corte del 31 de agosto",
    "licencia": "Libre Uso MX",
    "user_agent": UA,
    "crudos": {a: {"archivo": p.name, "bytes": p.stat().st_size} for a, p in archivos.items()},
    "agregados": {},
}
for nombre, df in salidas.items():
    ruta = DATA / f"{nombre}.csv"
    df.write_csv(ruta)
    manifiesto["agregados"][nombre] = {"filas": df.height, "columnas": df.columns}
(DATA / "manifest.json").write_text(json.dumps(manifiesto, ensure_ascii=False, indent=2))
pd.DataFrame({k: v["filas"] for k, v in manifiesto["agregados"].items()}, index=["filas"]).T

,filas
nacional,5
entidad_sexo,339
sector_sexo,107
rango_salarial,81
rango_uma,128
entidad_rango,2591
entidad_uma,4094
sexo_rango,171
tamano_patron,40
edad_sexo,148


## 8. Qué se puede afirmar con estos datos, y qué no

- **Sí:** salario base de cotización promedio y distribución por rangos, por estado, sector, sexo, edad y tamaño de patrón, para el empleo formal afiliado al IMSS.
- **Con cuidado:** el salario está topado a 25 UMA, así que cualquier promedio es una cota inferior y conviene acompañarlo de la mediana; el estado es el del registro patronal, no necesariamente donde trabaja la persona; las cifras son nominales.
- **No:** hablar de "los salarios en México" sin decir que esto cubre solo al empleo formal privado afiliado al IMSS, ni sumar `asegurados` como si fueran empleos.